# Ancient Greek neural voice benchmark\nUpload `ancient-greek-reader-mvp-v4.zip`. This notebook runs the same fixed Classical Attic benchmark through **Kokoro (raw Attic phonemes)** and **Meta MMS Ancient Greek**, then lets you listen side-by-side.\n\nKokoro is the main phoneme-controlled candidate; MMS is the dedicated Ancient Greek comparison baseline.

In [ ]:
from google.colab import files\nuploaded = files.upload()\nzip_name = next(iter(uploaded))\nprint('uploaded:', zip_name)

In [ ]:
import os, zipfile, pathlib, shutil\nwork = pathlib.Path('/content/agr')\nif work.exists(): shutil.rmtree(work)\nwork.mkdir()\nwith zipfile.ZipFile(zip_name) as z: z.extractall(work)\ncandidates = list(work.rglob('pyproject.toml'))\nbackend = candidates[0].parent\nprint('backend:', backend)\nos.chdir(backend)

In [ ]:
!pip -q install -e '.[kokoro,mms]'\nprint('Dependencies installed')

In [ ]:
!python scripts/run_voice_benchmark.py --providers kokoro mms --out ../benchmark-output-colab

In [ ]:
import json, pathlib\nfrom IPython.display import Audio, display, Markdown\nout = (backend / '../benchmark-output-colab').resolve()\nmanifest = json.loads((out/'manifest.json').read_text())\nfor row in manifest:\n    display(Markdown(f"### {row['id']} — {row['text']}\n`{row['ipa']}`"))\n    for provider, value in row['audio'].items():\n        if isinstance(value, str):\n            display(Markdown(f"**{provider}**"))\n            display(Audio(filename=str(out/value)))\n        else:\n            print(provider, value)

## What to listen for\nScore each model on two independent dimensions: **naturalness** and **Classical Attic fidelity**. Specifically listen for `b d g`, aspirated `pʰ tʰ kʰ`, `y`, long `ɛː/ɔː`, rough breathing, diphthongs, gamma nasalization, geminates, and whether the connected sentences sound usable for repeated study.

In [ ]:
archive = shutil.make_archive('/content/ancient-greek-voice-benchmark', 'zip', out)\nfiles.download(archive)